# Extracción de embeddings con ECAPA-TDNN

En esta etapa del proyecto se extraen embeddings de locutor utilizando el modelo **ECAPA-TDNN**.  
El objetivo es construir una segunda representación vectorial de los audios de la base de datos **CIEMPIESS**, para posteriormente compararla con los embeddings obtenidos previamente mediante **x-vector**.

La comparación entre ambos modelos permitirá analizar si una arquitectura más moderna, como ECAPA-TDNN, produce representaciones más discriminativas para la tarea de **verificación de locutor**.



## ¿Qué es ECAPA-TDNN?

**ECAPA-TDNN** significa:

**Emphasized Channel Attention, Propagation and Aggregation Time Delay Neural Network**

Es una arquitectura neuronal diseñada para extraer embeddings de locutor en tareas de reconocimiento y verificación de hablante.  
Al igual que x-vector, ECAPA-TDNN parte de una arquitectura basada en **TDNN**, pero incorpora mejoras que buscan capturar información más rica y robusta de la voz.

Entre sus componentes principales se encuentran:

- **Bloques TDNN**, que permiten modelar información temporal local de la señal de voz.
- **Mecanismos de atención por canal**, que ayudan al modelo a dar más importancia a las características acústicas más relevantes.
- **Conexiones residuales y agregación de información**, que permiten aprovechar representaciones generadas en distintas capas de la red.
- **Pooling estadístico atento**, que transforma una secuencia temporal de longitud variable en un vector fijo representativo del locutor.

En términos generales, ECAPA-TDNN recibe una señal de voz, extrae características acústicas internas y produce un vector numérico de dimensión fija conocido como **embedding de locutor**.



## ¿Por qué usar ECAPA-TDNN en este proyecto?

En la etapa anterior se extrajeron embeddings utilizando **x-vector**, una arquitectura clásica y ampliamente utilizada en reconocimiento de locutor.

Sin embargo, ECAPA-TDNN puede entenderse como una evolución de los sistemas basados en TDNN.  
Mientras que x-vector utiliza capas TDNN y pooling estadístico para obtener una representación fija del hablante, ECAPA-TDNN introduce mecanismos adicionales para mejorar la capacidad de discriminación entre locutores.

Por esta razón, en este proyecto se utilizarán ambos modelos bajo una lógica experimental similar:

1. Cargar los audios de la base CIEMPIESS.
2. Extraer embeddings con un modelo preentrenado.
3. Guardar los embeddings junto con sus identificadores.
4. Usar esos embeddings posteriormente para calcular similitudes entre pares de audios.
5. Comparar el desempeño de x-vector y ECAPA-TDNN mediante métricas de verificación de locutor.



## Modelo utilizado

Para esta fase se utilizará un modelo ECAPA-TDNN preentrenado disponible en **SpeechBrain**:

```python
speechbrain/spkrec-ecapa-voxceleb
```
Este modelo fue entrenado sobre datos de la familia VoxCeleb y puede utilizarse tanto para verificación de locutor como para extracción de embeddings.

En este proyecto no se entrenará ECAPA-TDNN desde cero.
Se utilizará como extractor de embeddings preentrenado, de forma similar a como se hizo previamente con x-vector.

## 01. Librerías principales de esta fase

En esta sección se importan las librerías necesarias para realizar la extracción de embeddings con **ECAPA-TDNN**.

Las librerías principales que se utilizarán son:

- **torch**: permite trabajar con tensores y ejecutar modelos neuronales.
- **torchaudio**: proporciona herramientas para procesamiento de audio con PyTorch.
- **pandas**: permite manipular tablas de metadatos, enrollment, test y trials.
- **numpy**: permite manipular arreglos numéricos y matrices de embeddings.
- **Path**: facilita el manejo de rutas de archivos de forma ordenada.
- **tqdm**: permite visualizar barras de progreso durante la extracción de embeddings.
- **EncoderClassifier**: clase de SpeechBrain utilizada para cargar modelos preentrenados de reconocimiento de locutor.

A diferencia de un flujo donde todos los audios están descargados localmente, en este proyecto **no se cuenta con la base CIEMPIESS descargada en el directorio de trabajo**.  

Para acceder a los audios se utilizará la librería **datasets**, siguiendo la lógica empleada en etapas anteriores del proyecto.



In [1]:
# Librerías principales para extracción de embeddings ECAPA-TDNN

# Librerías estándar
import os
import pickle
from pathlib import Path

# Manejo de datos
import numpy as np
import pandas as pd

# Deep learning y audio
import torch
import torchaudio

# Carga de audios desde datasets
from datasets import load_dataset, Audio
import soundfile as sf
import io

# Barra de progreso
from tqdm.auto import tqdm



In [2]:
# Modelo preentrenado de SpeechBrain para embeddings de locutor
from speechbrain.inference.speaker import EncoderClassifier

c:\Users\Diego Max.000\anaconda3\Lib\site-packages\speechbrain\utils\torch_audio_backend.py:57: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()


### Verificación del dispositivo de cómputo

La extracción de embeddings con ECAPA-TDNN puede realizarse en CPU o GPU.

Si se cuenta con una GPU compatible con CUDA, el proceso puede ser considerablemente más rápido.  
En caso contrario, el modelo se ejecutará en CPU.

Esta verificación permite definir de forma automática el dispositivo que utilizará PyTorch.

In [3]:
# Verificación de dispositivo: CPU o GPU

# Si hay una GPU disponible, PyTorch usará CUDA.
# En caso contrario, se usará CPU.
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Dispositivo seleccionado: {device}")

Dispositivo seleccionado: cpu


### Configuración de rutas de trabajo

En esta fase se utilizarán archivos generados en etapas anteriores, como:

- metadatos limpios de CIEMPIESS,
- particiones de enrollment y test,
- tabla de trials,
- y posteriormente los archivos donde se guardarán los embeddings ECAPA-TDNN.

Para mantener el proyecto ordenado, se definen rutas base y carpetas de salida.

In [4]:
# Configuración de rutas del proyecto


# Directorio base del proyecto.
# Como el notebook está dentro de la carpeta principal, usamos Path.cwd().
PROJECT_DIR = Path.cwd()

# Archivos principales del protocolo experimental
METADATA_PATH = PROJECT_DIR / "metadata_ciempiess_light_limpio_preliminar.csv"
ENROLLMENT_PATH = PROJECT_DIR / "enrollment.csv"
TEST_PATH = PROJECT_DIR / "test.csv"
TRIALS_PATH = PROJECT_DIR / "trials.csv"

# Carpeta general de salidas
OUTPUTS_DIR = PROJECT_DIR / "outputs"

# Carpeta general de embeddings
EMBEDDINGS_DIR = OUTPUTS_DIR / "embeddings"

# Carpeta específica para embeddings ECAPA-TDNN
ECAPA_EMBEDDINGS_DIR = EMBEDDINGS_DIR / "ecapa_tdnn"

# Carpeta para modelos preentrenados
PRETRAINED_MODELS_DIR = PROJECT_DIR / "pretrained_models"

# Carpeta específica para el modelo ECAPA-TDNN
ECAPA_MODEL_DIR = PRETRAINED_MODELS_DIR / "ecapa_tdnn"

# Crear carpetas necesarias si no existen
ECAPA_EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)
ECAPA_MODEL_DIR.mkdir(parents=True, exist_ok=True)


### Carga inicial de tablas

Se cargan las tablas principales del protocolo experimental:

- `metadata_df`: contiene información general de los audios.
- `enrollment_df`: contiene los audios usados como referencia de cada locutor.
- `test_df`: contiene los audios usados como pruebas.
- `trials_df`: contiene los pares de comparación para la etapa posterior de scoring.

En esta fase todavía no se calculan similitudes.  
Únicamente se preparan las tablas necesarias para extraer embeddings.

In [5]:
# Carga de metadatos y particiones experimentales


metadata_df = pd.read_csv(METADATA_PATH)
enrollment_df = pd.read_csv(ENROLLMENT_PATH)
test_df = pd.read_csv(TEST_PATH)
trials_df = pd.read_csv(TRIALS_PATH)

print("metadata_df:", metadata_df.shape)
print("enrollment_df:", enrollment_df.shape)
print("test_df:", test_df.shape)
print("trials_df:", trials_df.shape)


metadata_df: (13252, 8)
enrollment_df: (87, 9)
test_df: (13165, 9)
trials_df: (26330, 13)


## 02. Carga del modelo ECAPA-TDNN

En esta sección se carga el modelo preentrenado **ECAPA-TDNN** disponible en SpeechBrain.

El modelo utilizado será:

```python
speechbrain/spkrec-ecapa-voxceleb
```

In [6]:
# Carga del modelo ECAPA-TDNN preentrenado


# Nombre del modelo preentrenado disponible en SpeechBrain.
# Este modelo está diseñado para verificación de locutor y extracción de embeddings.
ECAPA_MODEL_SOURCE = "speechbrain/spkrec-ecapa-voxceleb"

# Carpeta local donde se almacenará el modelo descargado.
# Si el modelo ya existe, SpeechBrain lo reutilizará.
ECAPA_MODEL_SAVE_DIR = str(ECAPA_MODEL_DIR)

print("Modelo ECAPA-TDNN seleccionado:")
print(ECAPA_MODEL_SOURCE)

print("\nCarpeta local del modelo:")
print(ECAPA_MODEL_SAVE_DIR)

Modelo ECAPA-TDNN seleccionado:
speechbrain/spkrec-ecapa-voxceleb

Carpeta local del modelo:
c:\Users\Diego Max.000\Desktop\Servicio-Social-RN-Voz-Forense\CODIGOS\EXPERIMENTOS\pretrained_models\ecapa_tdnn


### Inicialización del modelo

La clase `EncoderClassifier` permite cargar modelos preentrenados de SpeechBrain para tareas de reconocimiento de locutor.

En este caso, el modelo ECAPA-TDNN se carga desde Hugging Face y se guarda localmente en la carpeta definida previamente.

El argumento `run_opts={"device": device}` indica si el modelo debe ejecutarse en CPU o GPU, dependiendo de la disponibilidad detectada en la sección anterior.

In [7]:
# Inicialización del extractor ECAPA-TDNN


# Cargar el modelo preentrenado.
# run_opts permite indicar si el modelo correrá en CPU o GPU.
ecapa_classifier = EncoderClassifier.from_hparams(
    source=ECAPA_MODEL_SOURCE,
    savedir=ECAPA_MODEL_SAVE_DIR,
    run_opts={"device": device}
)

print("Modelo ECAPA-TDNN cargado correctamente.")

c:\Users\Diego Max.000\anaconda3\Lib\site-packages\speechbrain\utils\torch_audio_backend.py:57: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()
c:\Users\Diego Max.000\anaconda3\Lib\site-packages\speechbrain\utils\autocast.py:188: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  wrapped_fwd = torch.cuda.amp.custom_fwd(fwd, cast_inputs=cast_inputs)
c:\Users\Diego Max.000\anaconda3\Lib\site-packages\speechbrain\utils\parameter_transfer.py:234: UserWarning: Requested Pretrainer collec

Modelo ECAPA-TDNN cargado correctamente.


## 03. Carga de audios desde `datasets`

En este proyecto no se cuenta con los archivos de audio de CIEMPIESS descargados localmente.

Por esta razón, la extracción de embeddings no se hará leyendo archivos `.wav` directamente desde una carpeta local, sino recuperando los audios mediante la librería `datasets`.

La base utilizada será:

```python
ciempiess/ciempiess_light
```

Sin embargo, no utilizaremos la decodificación automática de la columna `audio`, porque en algunos entornos esta opción puede requerir `torchcodec`.

Para evitar ese problema, configuramos la columna de audio con:

```python
Audio(decode=False)
```


In [8]:
# Carga de CIEMPIESS Light desde Hugging Face Datasets


# Nombre del dataset en Hugging Face
CIEMPIESS_DATASET_NAME = "ciempiess/ciempiess_light"

# Cargar split principal
ciempiess = load_dataset(
    CIEMPIESS_DATASET_NAME,
    split="train"
)

# Configurar la columna de audio SIN decodificación automática.
# Esto evita depender de torchcodec.
ciempiess = ciempiess.cast_column(
    "audio",
    Audio(decode=False)
)



### Creación de índice rápido 

Para recuperar audios de manera eficiente, se construye un diccionario que relaciona cada `audio_id` con su posición dentro del objeto `Dataset`.

Esto evita recorrer todo el dataset cada vez que se desea buscar un audio.

In [9]:
audio_id_to_index = {
    item["audio_id"]: idx
    for idx, item in enumerate(ciempiess)
}

print("Número de audio_id indexados:", len(audio_id_to_index))

Número de audio_id indexados: 16663


### Función `read_audio_from_hf_item`

La función read_audio_from_hf_item() recibe un registro individual del dataset de Hugging Face y devuelve la señal de audio como arreglo de NumPy.

Como la columna `audio` fue configurada con `decode=False`, el audio no se decodifica automáticamente.  
Por ello, esta función contempla dos casos:

1. El audio está disponible como bytes.
2. El audio está disponible mediante una ruta interna/cache.

En ambos casos, la lectura se realiza con `soundfile`, evitando el uso de `torchcodec`.

In [10]:
def read_audio_from_hf_item(item):
    """
    Lee la señal de audio desde un registro del dataset CIEMPIESS Light.

    La columna `audio` debe estar configurada con Audio(decode=False).
    Esto evita la decodificación automática y permite leer el archivo
    manualmente usando soundfile.

    Parámetros
    ----------
    item : dict
        Registro individual del dataset de Hugging Face.

    Retorna
    -------
    signal_np : np.ndarray
        Señal de audio como arreglo de NumPy.

    sampling_rate : int
        Frecuencia de muestreo del audio.
    """

    # Extraemos el diccionario asociado a la columna de audio.
    audio_info = item["audio"]

    # Obtenemos los bytes del audio, si están disponibles.
    audio_bytes = audio_info["bytes"]

    # Obtenemos la ruta interna/cache del audio, si está disponible.
    audio_path = audio_info["path"]

    # Caso 1: el audio está disponible como bytes.
    if audio_bytes is not None:
        signal_np, sampling_rate = sf.read(io.BytesIO(audio_bytes))

    # Caso 2: el audio se debe leer desde una ruta.
    else:
        signal_np, sampling_rate = sf.read(audio_path)

    return signal_np, sampling_rate

### Función `read_audio_by_id`

La función read_audio_by_id() permite recuperar la señal de audio a partir de un audio_id.

Esta función utiliza el diccionario audio_id_to_index, que relaciona cada identificador de audio con su posición dentro del dataset de Hugging Face.

Esta función será reutilizada tanto para x-vector como para ECAPA-TDNN, ya que el mecanismo de lectura del audio es independiente del modelo de embeddings.

In [11]:
def read_audio_by_id(audio_id, dataset, audio_id_to_index):
    """
    Lee la señal de audio correspondiente a un audio_id.

    Parámetros
    ----------
    audio_id : str
        Identificador único del audio.

    dataset : datasets.Dataset
        Dataset de Hugging Face con la columna audio en decode=False.

    audio_id_to_index : dict
        Diccionario que mapea audio_id a índice dentro del dataset.

    Retorna
    -------
    signal_np : np.ndarray
        Señal de audio en formato NumPy.

    sampling_rate : int
        Frecuencia de muestreo del audio.
    """

    # Verificar que el audio_id exista dentro del índice.
    if audio_id not in audio_id_to_index:
        raise ValueError(f"No se encontró audio_id en el dataset: {audio_id}")

    # Recuperar índice asociado al audio_id.
    item_index = audio_id_to_index[audio_id]

    # Extraer registro individual.
    item = dataset[item_index]

    # Leer señal de audio.
    signal_np, sampling_rate = read_audio_from_hf_item(item)

    return signal_np, sampling_rate

In [12]:
def prepare_signal_for_speechbrain(signal_np, device="cpu"):
    """
    Convierte una señal de audio en formato NumPy a un tensor de PyTorch
    compatible con modelos de SpeechBrain.

    Parámetros
    ----------
    signal_np : np.ndarray
        Señal de audio leída desde el dataset.

    device : str
        Dispositivo donde se colocará el tensor.
        Puede ser "cpu" o "cuda".

    Retorna
    -------
    signal_tensor : torch.Tensor
        Tensor de PyTorch con forma [1, T], listo para SpeechBrain.
    """

    # Convertir arreglo NumPy a tensor float32.
    signal_tensor = torch.tensor(signal_np, dtype=torch.float32)

    # Caso 1: audio mono con forma [T].
    if signal_tensor.ndim == 1:
        signal_tensor = signal_tensor.unsqueeze(0)

    # Caso 2: audio multicanal con forma [T, C].
    # Se promedian los canales para obtener una señal mono.
    elif signal_tensor.ndim == 2:
        signal_tensor = signal_tensor.mean(dim=1).unsqueeze(0)

    else:
        raise ValueError(f"Forma de señal no esperada: {signal_tensor.shape}")

    # Mover tensor al dispositivo seleccionado.
    signal_tensor = signal_tensor.to(device)

    return signal_tensor

## 04. Prueba de extracción de embedding ECAPA-TDNN con un solo audio

Antes de extraer embeddings para todos los audios de `enrollment` y `test`, primero se realiza una prueba con un solo audio.

Esta prueba tiene como objetivo verificar que todo el flujo funcione correctamente:

1. Recuperar un audio desde Hugging Face Datasets usando su `audio_id`.
2. Leer la señal manualmente con `soundfile`, evitando el uso de `torchcodec`.
3. Convertir la señal a un tensor compatible con SpeechBrain.
4. Enviar la señal al modelo ECAPA-TDNN.
5. Obtener un embedding de locutor.
6. Verificar la forma del embedding generado.

Este paso es importante porque permite detectar errores antes de procesar todos los audios del protocolo experimental.

In [13]:
# Selección de un audio de prueba desde enrollment


# Tomamos el primer audio del conjunto de enrollment.
# Este audio se usará únicamente para validar el flujo completo.
example_audio_id = enrollment_df.iloc[0]["audio_id"]

print("Audio de prueba seleccionado:")
print(example_audio_id)

Audio de prueba seleccionado:
CMPL_F_01_12MAB_00651


In [14]:
# Recuperamos la señal de audio usando la función reutilizada
# de la etapa anterior.
signal_np, sampling_rate = read_audio_by_id(
    audio_id=example_audio_id,
    dataset=ciempiess,
    audio_id_to_index=audio_id_to_index
)

print("Audio leído correctamente.")

Audio leído correctamente.


In [15]:
# Convertimos la señal NumPy a tensor de PyTorch con forma [1, T].
signal_tensor = prepare_signal_for_speechbrain(
    signal_np=signal_np,
    device=device
)

print("Tensor preparado correctamente.")

Tensor preparado correctamente.


### Extracción del embedding

Una vez que la señal tiene la forma correcta, se envía al modelo ECAPA-TDNN mediante el método:

```python
ecapa_classifier.encode_batch(signal_tensor)
```
Este método devuelve un tensor que contiene el embedding del locutor.
Como estamos en una fase de inferencia y no de entrenamiento, se utiliza:

```python
torch.no_grad()
```
Esto desactiva el cálculo de gradientes, reduce el uso de memoria y hace más eficiente la extracción.

In [16]:
# Extracción de embedding ECAPA-TDNN para un solo audio


with torch.no_grad():
    ecapa_embedding = ecapa_classifier.encode_batch(signal_tensor)

print("Embedding ECAPA-TDNN extraído correctamente.")

Embedding ECAPA-TDNN extraído correctamente.


In [17]:
# Conversión del embedding a NumPy

# squeeze(): elimina dimensiones unitarias.
# detach(): separa el tensor del grafo computacional.
# cpu(): mueve el tensor a CPU si estaba en GPU.
# numpy(): convierte el tensor a arreglo NumPy.
ecapa_embedding_np = ecapa_embedding.squeeze().detach().cpu().numpy()

print("Embedding convertido a NumPy.")

Embedding convertido a NumPy.


In [18]:
# Conversión del embedding a NumPy

# squeeze(): elimina dimensiones unitarias.
# detach(): separa el tensor del grafo computacional.
# cpu(): mueve el tensor a CPU si estaba en GPU.
# numpy(): convierte el tensor a arreglo NumPy.
ecapa_embedding_np = ecapa_embedding.squeeze().detach().cpu().numpy()

print("Embedding convertido a NumPy.")

Embedding convertido a NumPy.


In [19]:
# Validaciones básicas del embedding

# Verificamos que el embedding sea un vector unidimensional.
if ecapa_embedding_np.ndim != 1:
    raise ValueError(
        f"El embedding no es un vector 1D. "
        f"Forma obtenida: {ecapa_embedding_np.shape}"
    )

# Verificamos que no contenga valores NaN.
if np.isnan(ecapa_embedding_np).any():
    raise ValueError("El embedding contiene valores NaN.")

# Verificamos que no contenga valores infinitos.
if np.isinf(ecapa_embedding_np).any():
    raise ValueError("El embedding contiene valores infinitos.")

print("Validaciones completadas correctamente.")
print("Dimensión del embedding ECAPA-TDNN:", ecapa_embedding_np.shape[0])

Validaciones completadas correctamente.
Dimensión del embedding ECAPA-TDNN: 192


### Observación sobre la dimensión del embedding

El embedding generado por ECAPA-TDNN puede tener una dimensión distinta al embedding generado por x-vector.

Esto no representa un problema, ya que cada modelo produce su propia representación vectorial del locutor.

Lo importante es que:

- todos los embeddings ECAPA-TDNN tengan la misma dimensión entre sí;
- todos los embeddings x-vector tengan la misma dimensión entre sí;
- no se mezclen directamente embeddings de modelos distintos en una misma comparación.

En etapas posteriores, las similitudes se calcularán por separado:

- x-vector contra x-vector;
- ECAPA-TDNN contra ECAPA-TDNN.

## 05. Función para extraer embeddings ECAPA-TDNN desde `audio_id`

En la sección anterior se validó el flujo completo con un solo audio:

```text
audio_id → lectura desde Hugging Face Datasets → señal NumPy → tensor PyTorch → ECAPA-TDNN → embedding
```

Ahora encapsulamos ese proceso en una función reutilizable.

La función extract_ecapa_embedding_from_audio_id() recibe un audio_id, recupera el audio correspondiente y devuelve su embedding ECAPA-TDNN como arreglo de NumPy.

Esta función será utilizada posteriormente para extraer embeddings de todos los audios de enrollment y test.

In [20]:
# Función para extraer embedding ECAPA-TDNN desde audio_id


def extract_ecapa_embedding_from_audio_id(
    audio_id,
    dataset,
    audio_id_to_index,
    ecapa_classifier,
    device="cpu",
    expected_sampling_rate=16000
):
    """
    Parámetros
    ----------
    audio_id : str
        Identificador único del audio.

    dataset : datasets.Dataset
        Dataset de Hugging Face con la columna audio configurada con
        Audio(decode=False).

    audio_id_to_index : dict
        Diccionario que mapea cada audio_id con su índice dentro del dataset.

    ecapa_classifier : speechbrain.inference.speaker.EncoderClassifier
        Modelo ECAPA-TDNN preentrenado cargado con SpeechBrain.

    device : str, default="cpu"
        Dispositivo de cómputo donde se ejecutará el modelo.
        Puede ser "cpu" o "cuda".

    expected_sampling_rate : int, default=16000
        Frecuencia de muestreo esperada por el modelo.

    Retorna
    -------
    embedding_np : np.ndarray
        Embedding ECAPA-TDNN del audio, como vector NumPy 1D.

    sampling_rate : int
        Frecuencia de muestreo del audio procesado.
    """


    # Leer audio desde Hugging Face Datasets
    signal_np, sampling_rate = read_audio_by_id(
        audio_id=audio_id,
        dataset=dataset,
        audio_id_to_index=audio_id_to_index
    )


    # Validar frecuencia de muestreo
    if sampling_rate != expected_sampling_rate:
        raise ValueError(
            f"Frecuencia de muestreo inesperada para {audio_id}: "
            f"{sampling_rate}. Se esperaba {expected_sampling_rate}."
        )


    # Preparar tensor para SpeechBrain
    signal_tensor = prepare_signal_for_speechbrain(
        signal_np=signal_np,
        device=device
    )


    # Extraer embedding con ECAPA-TDNN
    ecapa_classifier.eval()

    with torch.no_grad():
        embedding = ecapa_classifier.encode_batch(signal_tensor)


    # Convertir embedding a NumPy
    embedding_np = embedding.squeeze().detach().cpu().numpy()


    # Validaciones básicas
    if embedding_np.ndim != 1:
        raise ValueError(
            f"El embedding de {audio_id} no es un vector 1D. "
            f"Forma obtenida: {embedding_np.shape}"
        )

    if np.isnan(embedding_np).any():
        raise ValueError(f"El embedding de {audio_id} contiene valores NaN.")

    if np.isinf(embedding_np).any():
        raise ValueError(f"El embedding de {audio_id} contiene valores infinitos.")

    return embedding_np, sampling_rate

## 06. Función general para extraer embeddings ECAPA-TDNN de un DataFrame

Después de validar la extracción de embeddings para un solo audio, el siguiente paso es automatizar el proceso para todos los audios de una tabla.

En este proyecto trabajaremos principalmente con dos conjuntos:

- `enrollment_df`: audios de referencia de cada locutor.
- `test_df`: audios que serán comparados contra los audios de enrollment.

La función que construiremos recibe un `DataFrame` con una columna `audio_id` y aplica la función:

```python
extract_ecapa_embedding_from_audio_id()

In [21]:
# Función general para extraer embeddings ECAPA-TDNN de un DataFrame


def extract_ecapa_embeddings_for_dataframe(
    df,
    dataset,
    audio_id_to_index,
    ecapa_classifier,
    device="cpu",
    audio_id_col="audio_id",
    expected_sampling_rate=16000
):
    """
    Extrae embeddings ECAPA-TDNN para todos los audios contenidos en un DataFrame.

    Parámetros
    ----------
    df : pd.DataFrame
        Tabla que contiene al menos una columna con identificadores de audio.

    dataset : datasets.Dataset
        Dataset de Hugging Face con la columna audio configurada con Audio(decode=False).

    audio_id_to_index : dict
        Diccionario que relaciona cada audio_id con su índice dentro del dataset.

    ecapa_classifier : speechbrain.inference.speaker.EncoderClassifier
        Modelo ECAPA-TDNN preentrenado.

    device : str, default="cpu"
        Dispositivo donde se realizará la inferencia: "cpu" o "cuda".

    audio_id_col : str, default="audio_id"
        Nombre de la columna que contiene los identificadores de audio.

    expected_sampling_rate : int, default=16000
        Frecuencia de muestreo esperada.

    Retorna
    -------
    embeddings_dict : dict
        Diccionario con la forma:

        {
            audio_id_1: embedding_1,
            audio_id_2: embedding_2,
            ...
        }

        donde cada embedding es un arreglo NumPy 1D.

    errors : list
        Lista con información de los audios que no pudieron procesarse.
        Cada elemento contiene el audio_id y el mensaje de error.
    """

    
    # Validar que exista la columna audio_id
    if audio_id_col not in df.columns:
        raise ValueError(
            f"No se encontró la columna '{audio_id_col}' en el DataFrame."
        )

    
    # Obtener lista única de audios
    audio_ids = df[audio_id_col].dropna().unique().tolist()

    print(f"Número de audios únicos a procesar: {len(audio_ids)}")

    # Diccionario donde se guardarán los embeddings
    embeddings_dict = {}

    # Lista para registrar errores sin detener todo el proceso
    errors = []

    # Poner el modelo en modo evaluación
    ecapa_classifier.eval()

    
    # Iterar sobre cada audio_id
    for audio_id in tqdm(audio_ids, desc="Extrayendo embeddings ECAPA-TDNN"):

        try:
            # Extraer embedding individual
            embedding_np, sampling_rate = extract_ecapa_embedding_from_audio_id(
                audio_id=audio_id,
                dataset=dataset,
                audio_id_to_index=audio_id_to_index,
                ecapa_classifier=ecapa_classifier,
                device=device,
                expected_sampling_rate=expected_sampling_rate
            )

            # Guardar embedding en el diccionario
            embeddings_dict[audio_id] = embedding_np

        except Exception as e:
            # Guardar información del error sin interrumpir todo el proceso
            errors.append({
                "audio_id": audio_id,
                "error": str(e)
            })

    print("Extracción finalizada.")
    print(f"Embeddings extraídos correctamente: {len(embeddings_dict)}")
    print(f"Errores encontrados: {len(errors)}")

    return embeddings_dict, errors

## 07. Extracción de embeddings ECAPA-TDNN para enrollment

En esta sección se extraen embeddings ECAPA-TDNN para los audios contenidos en `enrollment_df`.

El conjunto de enrollment contiene los audios de referencia asociados a cada locutor.  
En un sistema de verificación de locutor, estos audios representan la voz conocida del hablante contra la cual se compararán posteriormente los audios de prueba.

El flujo aplicado será:

```text
audio_id de enrollment → audio desde Hugging Face Datasets → ECAPA-TDNN → embedding

In [22]:
# Extracción de embeddings ECAPA-TDNN para enrollment

ecapa_enrollment_embeddings, ecapa_enrollment_errors = extract_ecapa_embeddings_for_dataframe(
    df=enrollment_df,
    dataset=ciempiess,
    audio_id_to_index=audio_id_to_index,
    ecapa_classifier=ecapa_classifier,
    device=device,
    audio_id_col="audio_id",
    expected_sampling_rate=16000
)

Número de audios únicos a procesar: 87


Extrayendo embeddings ECAPA-TDNN:   0%|          | 0/87 [00:00<?, ?it/s]

Extracción finalizada.
Embeddings extraídos correctamente: 87
Errores encontrados: 0


In [23]:
# Rutas de guardado para embeddings ECAPA-TDNN de enrollment


# Archivo principal: diccionario audio_id -> embedding
ECAPA_ENROLLMENT_PKL_PATH = ECAPA_EMBEDDINGS_DIR / "ecapa_enrollment_embeddings.pkl"

# Tabla resumen de embeddings disponibles
ECAPA_ENROLLMENT_SUMMARY_PATH = ECAPA_EMBEDDINGS_DIR / "ecapa_enrollment_summary.csv"

# Matriz NumPy de embeddings
ECAPA_ENROLLMENT_NPY_PATH = ECAPA_EMBEDDINGS_DIR / "ecapa_enrollment_embeddings.npy"

# Índice asociado a la matriz NumPy
ECAPA_ENROLLMENT_INDEX_PATH = ECAPA_EMBEDDINGS_DIR / "ecapa_enrollment_index.csv"

In [24]:
# Guardar diccionario de embeddings ECAPA-TDNN de enrollment


with open(ECAPA_ENROLLMENT_PKL_PATH, "wb") as f:
    pickle.dump(ecapa_enrollment_embeddings, f)

print("Diccionario de embeddings guardado en:")
print(ECAPA_ENROLLMENT_PKL_PATH)

Diccionario de embeddings guardado en:
c:\Users\Diego Max.000\Desktop\Servicio-Social-RN-Voz-Forense\CODIGOS\EXPERIMENTOS\outputs\embeddings\ecapa_tdnn\ecapa_enrollment_embeddings.pkl


### Construcción de matriz NumPy de embeddings

Además del diccionario, construiremos una matriz NumPy con todos los embeddings de enrollment.

La matriz tendrá forma:

```text
n_audios × dimensión_embedding

In [25]:
# Construcción de matriz NumPy e índice asociado

# Ordenamos los audio_id para que el resultado sea reproducible.
enrollment_audio_ids_ordered = sorted(ecapa_enrollment_embeddings.keys())

# Apilamos los embeddings siguiendo ese orden.
ecapa_enrollment_matrix = np.vstack([
    ecapa_enrollment_embeddings[audio_id]
    for audio_id in enrollment_audio_ids_ordered
])

# Construimos tabla índice.
ecapa_enrollment_index = pd.DataFrame({
    "row_index": np.arange(len(enrollment_audio_ids_ordered)),
    "audio_id": enrollment_audio_ids_ordered
})

# Agregamos speaker_id desde enrollment_df.
ecapa_enrollment_index = ecapa_enrollment_index.merge(
    enrollment_df[["audio_id", "speaker_id"]].drop_duplicates(),
    on="audio_id",
    how="left"
)

ecapa_enrollment_index["embedding_model"] = "ecapa_tdnn"
ecapa_enrollment_index["embedding_dim"] = ecapa_enrollment_matrix.shape[1]

In [26]:
# Guardar matriz NumPy e índice asociado

np.save(
    ECAPA_ENROLLMENT_NPY_PATH,
    ecapa_enrollment_matrix
)

ecapa_enrollment_index.to_csv(
    ECAPA_ENROLLMENT_INDEX_PATH,
    index=False
)


### Archivos generados para enrollment

Al finalizar esta sección se generan los siguientes archivos:

| Archivo | Descripción |
|---|---|
| `ecapa_enrollment_embeddings.pkl` | Diccionario que relaciona cada `audio_id` con su embedding ECAPA-TDNN. |
| `ecapa_enrollment_embeddings.npy` | Matriz NumPy con todos los embeddings de enrollment. |
| `ecapa_enrollment_index.csv` | Índice que relaciona cada fila de la matriz `.npy` con su `audio_id`. |

El archivo `.pkl` será especialmente útil para scoring basado en trials, mientras que la matriz `.npy` puede ser útil para análisis posteriores, visualización o experimentos adicionales.

## 08. Extracción de embeddings ECAPA-TDNN para test

En esta sección se extraen embeddings ECAPA-TDNN para los audios contenidos en `test_df`.

El conjunto `test` contiene los audios que serán comparados contra los audios de `enrollment` en la etapa de verificación de locutor.

El flujo aplicado será:

```text
audio_id de test → audio desde Hugging Face Datasets → ECAPA-TDNN → embedding
```

Este proceso utiliza la misma función general que se aplicó sobre `enrollment_df`:

```python
extract_ecapa_embeddings_for_dataframe()
```

In [32]:
ecapa_test_embeddings, ecapa_test_errors = extract_ecapa_embeddings_for_dataframe(
    df=test_df,
    dataset=ciempiess,
    audio_id_to_index=audio_id_to_index,
    ecapa_classifier=ecapa_classifier,
    device=device,
    audio_id_col="audio_id",
    expected_sampling_rate=16000
)

Número de audios únicos a procesar: 13165


Extrayendo embeddings ECAPA-TDNN:   0%|          | 0/13165 [00:00<?, ?it/s]

Extracción finalizada.
Embeddings extraídos correctamente: 13165
Errores encontrados: 0


## 09. Guardado de embeddings ECAPA-TDNN de test

En esta sección se guardan los embeddings ECAPA-TDNN generados para los audios de `test`.

Estos embeddings serán utilizados posteriormente en la etapa de scoring, donde cada audio de prueba será comparado contra los audios de enrollment definidos en los trials.

In [33]:
# Rutas de guardado para embeddings ECAPA-TDNN de test


# Archivo principal: diccionario audio_id -> embedding
ECAPA_TEST_PKL_PATH = ECAPA_EMBEDDINGS_DIR / "ecapa_test_embeddings.pkl"

# Matriz NumPy de embeddings
ECAPA_TEST_NPY_PATH = ECAPA_EMBEDDINGS_DIR / "ecapa_test_embeddings.npy"

# Índice asociado a la matriz NumPy
ECAPA_TEST_INDEX_PATH = ECAPA_EMBEDDINGS_DIR / "ecapa_test_index.csv"


### Guardado del diccionario de embeddings

Primero se guarda el diccionario `ecapa_test_embeddings` en formato `.pkl`.

Este archivo conserva directamente la relación entre cada `audio_id` y su embedding ECAPA-TDNN.

Este formato será especialmente útil en la siguiente etapa, porque los trials contienen pares de audios identificados por sus `audio_id`.

In [34]:
# Guardar diccionario de embeddings ECAPA-TDNN de test

with open(ECAPA_TEST_PKL_PATH, "wb") as f:
    pickle.dump(ecapa_test_embeddings, f)

print("Diccionario de embeddings de test guardado en:")
print(ECAPA_TEST_PKL_PATH)

Diccionario de embeddings de test guardado en:
c:\Users\Diego Max.000\Desktop\Servicio-Social-RN-Voz-Forense\CODIGOS\EXPERIMENTOS\outputs\embeddings\ecapa_tdnn\ecapa_test_embeddings.pkl


### Construcción de matriz NumPy de embeddings

Además del diccionario, se construye una matriz NumPy con todos los embeddings de test.

La matriz tendrá forma:

```text
n_audios_test × dimensión_embedding

In [35]:
# Construcción de matriz NumPy e índice asociado para test


# Ordenamos los audio_id para que el resultado sea reproducible.
test_audio_ids_ordered = sorted(ecapa_test_embeddings.keys())

# Apilamos los embeddings siguiendo ese orden.
ecapa_test_matrix = np.vstack([
    ecapa_test_embeddings[audio_id]
    for audio_id in test_audio_ids_ordered
])

# Construimos tabla índice.
ecapa_test_index = pd.DataFrame({
    "row_index": np.arange(len(test_audio_ids_ordered)),
    "audio_id": test_audio_ids_ordered
})

# Agregamos speaker_id desde test_df.
ecapa_test_index = ecapa_test_index.merge(
    test_df[["audio_id", "speaker_id"]].drop_duplicates(),
    on="audio_id",
    how="left"
)

ecapa_test_index["embedding_model"] = "ecapa_tdnn"
ecapa_test_index["embedding_dim"] = ecapa_test_matrix.shape[1]

In [36]:
# Guardar matriz NumPy e índice asociado para test

np.save(
    ECAPA_TEST_NPY_PATH,
    ecapa_test_matrix
)

ecapa_test_index.to_csv(
    ECAPA_TEST_INDEX_PATH,
    index=False
)

### Verificación de archivos guardados

Finalmente, se verifica que todos los archivos hayan sido creados correctamente.

Esta verificación ayuda a confirmar que los embeddings ECAPA-TDNN de test quedaron almacenados y podrán reutilizarse en la etapa posterior de scoring.

In [37]:
# Verificación de archivos guardados para test


saved_test_files = {
    "Diccionario PKL": ECAPA_TEST_PKL_PATH,
    "Matriz NPY": ECAPA_TEST_NPY_PATH,
    "Índice CSV": ECAPA_TEST_INDEX_PATH
}

for name, path in saved_test_files.items():
    if path.exists():
        print(f"[OK] {name}: {path}")
    else:
        print(f"[ERROR] No se encontró {name}: {path}")

[OK] Diccionario PKL: c:\Users\Diego Max.000\Desktop\Servicio-Social-RN-Voz-Forense\CODIGOS\EXPERIMENTOS\outputs\embeddings\ecapa_tdnn\ecapa_test_embeddings.pkl
[OK] Matriz NPY: c:\Users\Diego Max.000\Desktop\Servicio-Social-RN-Voz-Forense\CODIGOS\EXPERIMENTOS\outputs\embeddings\ecapa_tdnn\ecapa_test_embeddings.npy
[OK] Índice CSV: c:\Users\Diego Max.000\Desktop\Servicio-Social-RN-Voz-Forense\CODIGOS\EXPERIMENTOS\outputs\embeddings\ecapa_tdnn\ecapa_test_index.csv


### Archivos generados para test

Al finalizar esta sección se generan los siguientes archivos:

| Archivo | Descripción |
|---|---|
| `ecapa_test_embeddings.pkl` | Diccionario que relaciona cada `audio_id` de test con su embedding ECAPA-TDNN. |
| `ecapa_test_embeddings.npy` | Matriz NumPy con todos los embeddings de test. |
| `ecapa_test_index.csv` | Índice que relaciona cada fila de la matriz `.npy` con su `audio_id`. |

Estos archivos son equivalentes a los generados para enrollment, pero corresponden al conjunto de prueba.

## 10. Verificación final de embeddings ECAPA-TDNN

Después de extraer y guardar los embeddings ECAPA-TDNN, es necesario verificar que los archivos generados puedan cargarse correctamente.

Esta sección tiene como objetivo validar que:

1. Los archivos esperados existen en la carpeta de salida.
2. Los diccionarios `.pkl` pueden cargarse nuevamente.
3. Las matrices `.npy` tienen dimensiones consistentes.
4. Los archivos índice `.csv` corresponden con las filas de las matrices.
5. Los embeddings de enrollment y test tienen la misma dimensión.
6. Los audios utilizados en los trials cuentan con embeddings disponibles.

Esta verificación es importante porque los embeddings guardados serán la entrada directa para la etapa posterior de scoring.

In [38]:
# ============================================================
# Verificación final de archivos generados
# ============================================================

expected_ecapa_files = {
    "Enrollment embeddings PKL": ECAPA_ENROLLMENT_PKL_PATH,
    "Enrollment embeddings NPY": ECAPA_ENROLLMENT_NPY_PATH,
    "Enrollment index CSV": ECAPA_ENROLLMENT_INDEX_PATH,
    
    "Test embeddings PKL": ECAPA_TEST_PKL_PATH,
    "Test embeddings NPY": ECAPA_TEST_NPY_PATH,
    "Test index CSV": ECAPA_TEST_INDEX_PATH,
}

for name, path in expected_ecapa_files.items():
    if path.exists():
        print(f"[OK] {name}: {path}")
    else:
        print(f"[ERROR] No se encontró {name}: {path}")

[OK] Enrollment embeddings PKL: c:\Users\Diego Max.000\Desktop\Servicio-Social-RN-Voz-Forense\CODIGOS\EXPERIMENTOS\outputs\embeddings\ecapa_tdnn\ecapa_enrollment_embeddings.pkl
[OK] Enrollment embeddings NPY: c:\Users\Diego Max.000\Desktop\Servicio-Social-RN-Voz-Forense\CODIGOS\EXPERIMENTOS\outputs\embeddings\ecapa_tdnn\ecapa_enrollment_embeddings.npy
[OK] Enrollment index CSV: c:\Users\Diego Max.000\Desktop\Servicio-Social-RN-Voz-Forense\CODIGOS\EXPERIMENTOS\outputs\embeddings\ecapa_tdnn\ecapa_enrollment_index.csv
[OK] Test embeddings PKL: c:\Users\Diego Max.000\Desktop\Servicio-Social-RN-Voz-Forense\CODIGOS\EXPERIMENTOS\outputs\embeddings\ecapa_tdnn\ecapa_test_embeddings.pkl
[OK] Test embeddings NPY: c:\Users\Diego Max.000\Desktop\Servicio-Social-RN-Voz-Forense\CODIGOS\EXPERIMENTOS\outputs\embeddings\ecapa_tdnn\ecapa_test_embeddings.npy
[OK] Test index CSV: c:\Users\Diego Max.000\Desktop\Servicio-Social-RN-Voz-Forense\CODIGOS\EXPERIMENTOS\outputs\embeddings\ecapa_tdnn\ecapa_test_inde

### Carga de archivos guardados

Para comprobar que los archivos fueron guardados correctamente, se vuelven a cargar desde disco.

Esto simula lo que ocurrirá en la siguiente etapa del proyecto, donde ya no será necesario volver a ejecutar ECAPA-TDNN, sino únicamente cargar los embeddings previamente generados.

In [39]:
# Cargar diccionarios de embeddings desde archivos PKL


with open(ECAPA_ENROLLMENT_PKL_PATH, "rb") as f:
    loaded_ecapa_enrollment_embeddings = pickle.load(f)

with open(ECAPA_TEST_PKL_PATH, "rb") as f:
    loaded_ecapa_test_embeddings = pickle.load(f)

print("Diccionarios cargados correctamente.")
print("Embeddings enrollment:", len(loaded_ecapa_enrollment_embeddings))
print("Embeddings test:", len(loaded_ecapa_test_embeddings))

Diccionarios cargados correctamente.
Embeddings enrollment: 87
Embeddings test: 13165


In [40]:
# Cargar matrices NumPy e índices CSV


loaded_ecapa_enrollment_matrix = np.load(ECAPA_ENROLLMENT_NPY_PATH)
loaded_ecapa_test_matrix = np.load(ECAPA_TEST_NPY_PATH)

loaded_ecapa_enrollment_index = pd.read_csv(ECAPA_ENROLLMENT_INDEX_PATH)
loaded_ecapa_test_index = pd.read_csv(ECAPA_TEST_INDEX_PATH)

print("Matriz enrollment:", loaded_ecapa_enrollment_matrix.shape)
print("Matriz test:", loaded_ecapa_test_matrix.shape)

print("\nÍndice enrollment:")
display(loaded_ecapa_enrollment_index.head())

print("\nÍndice test:")
display(loaded_ecapa_test_index.head())

Matriz enrollment: (87, 192)
Matriz test: (13165, 192)

Índice enrollment:


,row_index,audio_id,speaker_id,embedding_model,embedding_dim
0,0,CMPL_F_01_12MAB_00651,F_01,ecapa_tdnn,192
1,1,CMPL_F_02_02MAB_00241,F_02,ecapa_tdnn,192
2,2,CMPL_F_03_09MAB_00142,F_03,ecapa_tdnn,192
3,3,CMPL_F_04_12ANG_00215,F_04,ecapa_tdnn,192
4,4,CMPL_F_05_01CAR_00021,F_05,ecapa_tdnn,192



Índice test:


,row_index,audio_id,speaker_id,embedding_model,embedding_dim
0,0,CMPL_F_01_03MAB_00001,F_01,ecapa_tdnn,192
1,1,CMPL_F_01_03MAB_00002,F_01,ecapa_tdnn,192
2,2,CMPL_F_01_03MAB_00003,F_01,ecapa_tdnn,192
3,3,CMPL_F_01_03MAB_00004,F_01,ecapa_tdnn,192
4,4,CMPL_F_01_03MAB_00005,F_01,ecapa_tdnn,192


### Validación entre matrices e índices

Cada fila de una matriz `.npy` representa el embedding de un audio.

Como NumPy no conserva identificadores, usamos archivos índice `.csv` para saber qué `audio_id` corresponde a cada fila.

Por lo tanto, el número de filas de cada matriz debe coincidir con el número de filas de su índice correspondiente.

In [41]:
# Validar correspondencia entre matrices e índices


if loaded_ecapa_enrollment_matrix.shape[0] != loaded_ecapa_enrollment_index.shape[0]:
    raise ValueError(
        "La matriz de enrollment y su índice no tienen el mismo número de filas."
    )

if loaded_ecapa_test_matrix.shape[0] != loaded_ecapa_test_index.shape[0]:
    raise ValueError(
        "La matriz de test y su índice no tienen el mismo número de filas."
    )

print("Las matrices y sus índices tienen tamaños compatibles.")

Las matrices y sus índices tienen tamaños compatibles.
